<a href="https://colab.research.google.com/github/ThanhVanLe0605/Data-Mining-For-Business-Analytics-In-Python/blob/main/Exercise/Ex_Chapter13__Combining_Methods_Ensembling_And_Lifting_Modeling_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SUMMARY

  * **13.1. Acceptance of Consumer Loan**
  * **13.2. eBay Auctions — Boosting and Bagging**
  * **13.3. Predicting Delayed Flights (Boosting)**
  * **13.4. Hair Care Product — Uplift Modeling**

# 13.1. Acceptance of Consumer Loan

In [120]:
# Import libraries
import pandas as pd
import numpy as np

## Dataset Overview
  * Objective: Predict the probability of a customer accepting a personal loan (PersonalLoan) based on their demographic profile and financial behavior.

  * Number of Features: 14 columns (including ID, predictor variables, and the target variable).

  * Target Variable: PersonalLoan (Binary: 0 = Rejection, 1 = Acceptance).

## Attribute Details
### 1. Demographic Information
  * ID: Unique identifier for each customer.

  * Age: Customer's age (ranges from 25 to 45 in the sample).

  * Experience: Years of professional work experience.

  * ZIP Code: Postal code of the customer's residence.

  * Family: Family size (ranging from 1 to 4 members).

  * Education: Education level (encoded numerically, typically: 1 = Undergraduate, 2 = Graduate, 3 = Advanced/Professional).
### 2. Financial Indicators
  * Income: Annual income of the customer (in thousands of USD).

  * CCAvg: Average monthly credit card spending (in thousands of USD).

Mortgage: Value of the house mortgage, if any (0 if none).
### 3. Banking Relations

  * SecuritiesAccount: Whether the customer has a securities account with the bank (1 = Yes, 0 = No).

  * CDAccount: Whether the customer has a certificate of deposit (CD) account (1 = Yes, 0 = No).

  * Online: Whether the customer uses internet banking services (1 = Yes, 0 = No).

  * CreditCard: Whether the customer uses a credit card issued by this bank (1 = Yes, 0 = No).

Universal Bank has begun a program to encourage its existing customers to borrow via a consumer loan program. The bank has promoted the loan to 5000 customers, of whom 480 accepted the offer. The data are available in file UniversalBank.csv. The bank now wants to develop a model to predict which customers have the greatest probability of accepting the loan, to reduce promotion costs and send the offer only to a subset of its customers.

We will develop several models, then combine them in an ensemble. The models we will use are (1) logistic regression, (2) $k$-nearest neighbors with $k = 3$, and (3) classification trees. Preprocess the data as follows:

  * Bin the following variables so they can be used in Naive Bayes: Age (5 bins), Experience (10 bins), Income (5 bins), CC Average (6 bins), and Mortgage (10 bins).

  * Education and Family can be used as is, without binning.

  * Zip code can be ignored.

  * Use one-hot-encoding to convert the categorical data into indicator variables.

  * Partition the data: 60% training, 40% validation.

  **a.** Fit models to the data for (1) logistic regression, (2) $k$-nearest neighbors with $k = 3$, (3) classification trees, and (4) Naive Bayes. Use Personal Loan as the outcome variable. Report the validation confusion matrix for each of the models.  

  **b.** Create a data frame with the actual outcome, predicted outcome of each of the models. Report the first 10 rows of this data frame

  **c.** Add two columns to this data frame for (1) a majority vote of predicted outcomes and (2) the average of the predicted probabilities. Using the classifications generated by these two methods derive a confusion matrix for each method and report the overall accuracy.

  **d.** Compare the error rates for the four individual methods and the two ensemble methods.

In [121]:
# Import data
df_bank = pd.read_csv('UniversalBank.csv')
df_bank.head()

,ID,Age,Experience,Income,ZIP Code,Family,CCAvg,Education,Mortgage,PersonalLoan,SecuritiesAccount,CDAccount,Online,CreditCard
0,1,25,1,49,91107,4,1.6,1,0,0,1,0,0,0
1,2,45,19,34,90089,3,1.5,1,0,0,1,0,0,0
2,3,39,15,11,94720,1,1.0,1,0,0,0,0,0,0
3,4,35,9,100,94112,1,2.7,2,0,0,0,0,0,0
4,5,35,8,45,91330,4,1.0,2,0,0,0,0,0,1


## PREPROCESSING UniversalBank.csv

In [123]:
# Data Transformation in preprocessing data : Discremination and Binning
# Bin the following variables so they can be used in Naive Baye
df_bank[['Age', 'Experience', 'Income', 'CCAvg','Mortgage']].describe()

,Age,Experience,Income,CCAvg,Mortgage
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,45.338400,20.104600,73.774200,1.937938,56.498800
std,11.463166,11.467954,46.033729,1.747659,101.713802
min,23.000000,-3.000000,8.000000,0.000000,0.000000
25%,35.000000,10.000000,39.000000,0.700000,0.000000
50%,45.000000,20.000000,64.000000,1.500000,0.000000
75%,55.000000,30.000000,98.000000,2.500000,101.000000
max,67.000000,43.000000,224.000000,10.000000,635.000000


In [124]:
# min Experience = -3
df_bank['Experience'] = df_bank['Experience'].clip(lower = 0)

# Check
df_bank['Experience'].min()

0

In [125]:
# Age (5 bins)
age_labels = ['Starting_Out', 'Family_Building', 'Established', 'Mature', 'Retirement_Phase']
df_bank['Age_Group'] = pd.cut(df_bank['Age'], bins = 5, labels = age_labels, right = True)
df_bank['Age_Group_Values'] = pd.cut(df_bank['Age'], bins = 5, right = True)

display(pd.Series(df_bank['Age_Group'].value_counts()))
display(pd.Series(df_bank['Age_Group'].cat.categories))
display(pd.Series(df_bank['Age_Group_Values'].cat.categories))
display(df_bank[['Age','Age_Group','Age_Group_Values']].head(10))

,count
Age_Group,
Mature,1202
Established,1132
Family_Building,1111
Retirement_Phase,806
Starting_Out,749


,0
0,Starting_Out
1,Family_Building
2,Established
3,Mature
4,Retirement_Phase


,0
0,"(22.956, 31.8]"
1,"(31.8, 40.6]"
2,"(40.6, 49.4]"
3,"(49.4, 58.2]"
4,"(58.2, 67.0]"


,Age,Age_Group,Age_Group_Values
0,25,Starting_Out,"(22.956, 31.8]"
1,45,Established,"(40.6, 49.4]"
2,39,Family_Building,"(31.8, 40.6]"
3,35,Family_Building,"(31.8, 40.6]"
4,35,Family_Building,"(31.8, 40.6]"
5,37,Family_Building,"(31.8, 40.6]"
6,53,Mature,"(49.4, 58.2]"
7,50,Mature,"(49.4, 58.2]"
8,35,Family_Building,"(31.8, 40.6]"
9,34,Family_Building,"(31.8, 40.6]"


In [126]:
# Experience(10 )
experience_labels = [
    'Junior_1', 'Junior_2',
    'Senior_1', 'Senior_2',
    'Lead_1', 'Lead_2',
    'Director_1', 'Director_2',
    'Expert_1', 'Expert_2'
]

min_exp = df_bank['Experience'].min()
max_exp = df_bank['Experience'].max()
experience_bins = np.linspace(min_exp, max_exp, 11)

df_bank['Experience_Group'] = pd.cut(df_bank['Experience'], bins = experience_bins, labels = experience_labels, right=True)
df_bank['Experience_Group_Values'] = pd.cut(df_bank['Experience'], bins = experience_bins, right=True)

display(df_bank['Experience_Group'].value_counts())
display(pd.Series(df_bank['Experience_Group'] .cat.categories))
display(pd.Series(df_bank['Experience_Group_Values'] .cat.categories))

display(df_bank[['Experience','Experience_Group','Experience_Group_Values']].head(10))


,count
Experience_Group,
Director_1,647
Senior_2,615
Lead_2,541
Lead_1,533
Junior_2,505
Director_2,500
Senior_1,483
Expert_1,461
Junior_1,401


,0
0,Junior_1
1,Junior_2
2,Senior_1
3,Senior_2
4,Lead_1
5,Lead_2
6,Director_1
7,Director_2
8,Expert_1
9,Expert_2


,0
0,"(0.0, 4.3]"
1,"(4.3, 8.6]"
2,"(8.6, 12.9]"
3,"(12.9, 17.2]"
4,"(17.2, 21.5]"
5,"(21.5, 25.8]"
6,"(25.8, 30.1]"
7,"(30.1, 34.4]"
8,"(34.4, 38.7]"
9,"(38.7, 43.0]"


,Experience,Experience_Group,Experience_Group_Values
0,1,Junior_1,"(0.0, 4.3]"
1,19,Lead_1,"(17.2, 21.5]"
2,15,Senior_2,"(12.9, 17.2]"
3,9,Senior_1,"(8.6, 12.9]"
4,8,Junior_2,"(4.3, 8.6]"
5,13,Senior_2,"(12.9, 17.2]"
6,27,Director_1,"(25.8, 30.1]"
7,24,Lead_2,"(21.5, 25.8]"
8,10,Senior_1,"(8.6, 12.9]"
9,9,Senior_1,"(8.6, 12.9]"


In [127]:
# Income (5 bins)
# CC Average (6 bins)
income_labels = ['Mass', 'Bronze', 'Silver', 'Gold', 'Diamond']
df_bank['Income_Group'] = pd.qcut(df_bank['Income'], q=5, labels=income_labels)
df_bank['Income_Group_Values']  = pd.qcut(df_bank['Income'], q=5)

display(df_bank['Income_Group'].value_counts())
display(pd.Series(df_bank['Income_Group'] .cat.categories))
display(pd.Series(df_bank['Income_Group_Values'].cat.categories))

display(df_bank[['Income','Income_Group','Income_Group_Values']].head(10))

,count
Income_Group,
Mass,1029
Silver,1017
Gold,1002
Diamond,979
Bronze,973


,0
0,Mass
1,Bronze
2,Silver
3,Gold
4,Diamond


,0
0,"(7.999, 33.0]"
1,"(33.0, 52.0]"
2,"(52.0, 78.0]"
3,"(78.0, 113.0]"
4,"(113.0, 224.0]"


,Income,Income_Group,Income_Group_Values
0,49,Bronze,"(33.0, 52.0]"
1,34,Bronze,"(33.0, 52.0]"
2,11,Mass,"(7.999, 33.0]"
3,100,Gold,"(78.0, 113.0]"
4,45,Bronze,"(33.0, 52.0]"
5,29,Mass,"(7.999, 33.0]"
6,72,Silver,"(52.0, 78.0]"
7,22,Mass,"(7.999, 33.0]"
8,81,Gold,"(78.0, 113.0]"
9,180,Diamond,"(113.0, 224.0]"


In [128]:
ccavg_labels = ['Low', 'Below_Average', 'Average', 'Above_Average', 'High', 'Very_High']

df_bank['CCAvg_Group'] = pd.qcut(df_bank['CCAvg'],  labels=ccavg_labels, q = 6)
df_bank['CCAvg_Group_Values'] = pd.qcut(df_bank['CCAvg'], q=6)

display(df_bank['CCAvg_Group'].value_counts())
display(pd.Series(df_bank['CCAvg_Group'] .cat.categories))
display(pd.Series(df_bank['CCAvg_Group_Values'].cat.categories))

display(df_bank[['CCAvg','CCAvg_Group','CCAvg_Group_Values']].head(10))

,count
CCAvg_Group,
Low,913
Average,832
Above_Average,832
Very_High,829
High,824
Below_Average,770


,0
0,Low
1,Below_Average
2,Average
3,Above_Average
4,High
5,Very_High


,0
0,"(-0.001, 0.4]"
1,"(0.4, 0.9]"
2,"(0.9, 1.5]"
3,"(1.5, 2.1]"
4,"(2.1, 3.1]"
5,"(3.1, 10.0]"


,CCAvg,CCAvg_Group,CCAvg_Group_Values
0,1.6,Above_Average,"(1.5, 2.1]"
1,1.5,Average,"(0.9, 1.5]"
2,1.0,Average,"(0.9, 1.5]"
3,2.7,High,"(2.1, 3.1]"
4,1.0,Average,"(0.9, 1.5]"
5,0.4,Low,"(-0.001, 0.4]"
6,1.5,Average,"(0.9, 1.5]"
7,0.3,Low,"(-0.001, 0.4]"
8,0.6,Below_Average,"(0.4, 0.9]"
9,8.9,Very_High,"(3.1, 10.0]"


In [133]:
# Mortgage (10 bins)
df_bank['Mortgage_Group'] = pd.qcut(df_bank['Mortgage'], q=10)

ValueError: Bin edges must be unique: Index([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 78.0, 123.0, 200.0, 635.0], dtype='float64', name='Mortgage').
You can drop duplicate edges by setting the 'duplicates' kwarg

In [135]:
yes_mort = df_bank.loc[df_bank['Mortgage'] > 0, 'Mortgage']

len(yes_mort)

1538

In [138]:
mort_bins = pd.qcut(yes_mort, q = 10, labels = [str(i) for i in range(1,11)])
mort_values = pd.qcut(yes_mort, q = 10)

mort_labels_mapped = {str(i): f'Debt_Tier_{i}' for i in range(1, 11)}
mort_labels_mapped

{'1': 'Debt_Tier_1',
 '2': 'Debt_Tier_2',
 '3': 'Debt_Tier_3',
 '4': 'Debt_Tier_4',
 '5': 'Debt_Tier_5',
 '6': 'Debt_Tier_6',
 '7': 'Debt_Tier_7',
 '8': 'Debt_Tier_8',
 '9': 'Debt_Tier_9',
 '10': 'Debt_Tier_10'}

In [139]:
mort_bins_custom = mort_bins.astype('category').cat.rename_categories(mort_labels_mapped)
mort_bins_custom.cat.categories

Index(['Debt_Tier_1', 'Debt_Tier_2', 'Debt_Tier_3', 'Debt_Tier_4',
       'Debt_Tier_5', 'Debt_Tier_6', 'Debt_Tier_7', 'Debt_Tier_8',
       'Debt_Tier_9', 'Debt_Tier_10'],
      dtype='object')

In [141]:
mort_bins_custom = mort_bins_custom.cat.add_categories('No_Mortgage')

df_bank['Mortgage_Group'] = mort_bins_custom
df_bank['Mortgage_Group'] = df_bank['Mortgage_Group'].fillna('No_Mortgage')
df_bank['Mortgage_Group_Values'] = mort_values

In [142]:
display(df_bank['Mortgage_Group'].value_counts())
display(pd.Series(df_bank['Mortgage_Group'] .cat.categories))
display(pd.Series(df_bank['Mortgage_Group_Values'].cat.categories))

display(df_bank[['Mortgage','CCAvg_Group','Mortgage_Group_Values']].head(10))

,count
Mortgage_Group,
No_Mortgage,3462
Debt_Tier_1,164
Debt_Tier_5,160
Debt_Tier_7,157
Debt_Tier_9,155
Debt_Tier_3,154
Debt_Tier_10,153
Debt_Tier_4,152
Debt_Tier_8,150


,0
0,Debt_Tier_1
1,Debt_Tier_2
2,Debt_Tier_3
3,Debt_Tier_4
4,Debt_Tier_5
5,Debt_Tier_6
6,Debt_Tier_7
7,Debt_Tier_8
8,Debt_Tier_9
9,Debt_Tier_10


,0
0,"(74.999, 89.0]"
1,"(89.0, 102.0]"
2,"(102.0, 116.0]"
3,"(116.0, 132.0]"
4,"(132.0, 153.0]"
5,"(153.0, 177.2]"
6,"(177.2, 209.0]"
7,"(209.0, 248.6]"
8,"(248.6, 323.0]"
9,"(323.0, 635.0]"


,Mortgage,CCAvg_Group,Mortgage_Group_Values
0,0,Above_Average,NaN
1,0,Average,NaN
2,0,Average,NaN
3,0,High,NaN
4,0,Average,NaN
5,155,Low,"(153.0, 177.2]"
6,0,Average,NaN
7,0,Low,NaN
8,104,Below_Average,"(102.0, 116.0]"
9,0,Very_High,NaN


In [144]:
df_bank.columns

Index(['ID', 'Age', 'Experience', 'Income', 'ZIP Code', 'Family', 'CCAvg',
       'Education', 'Mortgage', 'PersonalLoan', 'SecuritiesAccount',
       'CDAccount', 'Online', 'CreditCard', 'Age_Group', 'Age_Group_Values',
       'Experience_Group', 'Experience_Group_Values', 'Income_Group',
       'Income_Group_Values', 'CCAvg_Group', 'CCAvg_Group_Values',
       'Mortgage_Group', 'Mortgage_Group_Values'],
      dtype='object')

In [145]:
df_bank.head()

,ID,Age,Experience,Income,ZIP Code,Family,CCAvg,Education,Mortgage,PersonalLoan,...,Age_Group,Age_Group_Values,Experience_Group,Experience_Group_Values,Income_Group,Income_Group_Values,CCAvg_Group,CCAvg_Group_Values,Mortgage_Group,Mortgage_Group_Values
0,1,25,1,49,91107,4,1.6,1,0,0,...,Starting_Out,"(22.956, 31.8]",Junior_1,"(0.0, 4.3]",Bronze,"(33.0, 52.0]",Above_Average,"(1.5, 2.1]",No_Mortgage,NaN
1,2,45,19,34,90089,3,1.5,1,0,0,...,Established,"(40.6, 49.4]",Lead_1,"(17.2, 21.5]",Bronze,"(33.0, 52.0]",Average,"(0.9, 1.5]",No_Mortgage,NaN
2,3,39,15,11,94720,1,1.0,1,0,0,...,Family_Building,"(31.8, 40.6]",Senior_2,"(12.9, 17.2]",Mass,"(7.999, 33.0]",Average,"(0.9, 1.5]",No_Mortgage,NaN
3,4,35,9,100,94112,1,2.7,2,0,0,...,Family_Building,"(31.8, 40.6]",Senior_1,"(8.6, 12.9]",Gold,"(78.0, 113.0]",High,"(2.1, 3.1]",No_Mortgage,NaN
4,5,35,8,45,91330,4,1.0,2,0,0,...,Family_Building,"(31.8, 40.6]",Junior_2,"(4.3, 8.6]",Bronze,"(33.0, 52.0]",Average,"(0.9, 1.5]",No_Mortgage,NaN


### THE BUSINESS LOGIC OF DATA DISCRETIZATION IN MACHINE LEARNING

#### INTRODUCTION
In the era of Artificial Intelligence, we tend to collect data with as much precision as possible. Continuous variables like an exact age of 23, or a precise income of \$114,500, give a sense of absolute accuracy. However, for mathematical and machine learning models, this hyper-granularity often acts as a form of "noise" that hinders pattern recognition.

This is why data discretization—the process of binning continuous numbers into structured behavioral groups—emerges as a mandatory bridge to unlock the true predictive power of algorithms.

---

####  1. WHY DO WE BIN DATA? THE PARADOX OF "LOSING INFORMATION"

When we transform a dataset column from a continuous numerical format into categorical groups (e.g., converting exact ages into *Young*, *Family*, or *Senior*), we are technically **losing information** from a mathematical standpoint. The computer no longer knows whether a customer is exactly 24 or 28 years old; it only knows they belong to the *Young* group.

However, data science operates on a fundamental philosophy: **"Lose granular noise to gain higher-value business signals."**



* **Eliminating Overfitting and Noise**
  * A customer who is 31 years old and another who is 32 years old share virtually identical financial behaviors.
  * If we force a model to learn from raw, unbinned numbers, it will exhaustively try to find a statistical justification for the meaningless variance between 31 and 32.
  * Grouping them into a unified "Family Building" stage deletes this superficial noise and forces the computer to focus on the macroeconomic pattern.
* **Boosting Model Robustness against Outliers**
  * Real-world banking data is plagued by extreme outliers—such as a single ultra-high-net-worth individual earning 100 times more than the average population.
  * If left as a raw number, this outlier can warp the decision boundaries of many algorithms.
  * By capturing them safely inside a broader "High Income" bucket, the destructive mathematical variance of the outlier is neutralized, making the final model significantly more stable.

---

####  2. HOW MACHINE LEARNING BENEFITS (THE CASE OF NAIVE BAYES)

Machine learning algorithms, particularly **Naive Bayes**, operate entirely on the principles of conditional probability. It makes predictions by counting how many historical events occurred out of a specific sample pool.

* **The Continuous Data Trap for Naive Bayes**
  * If you ask a Naive Bayes model: *"What is the exact probability that a customer earning exactly \$91,345 will accept a personal loan?"*, the algorithm will scan the historical records.
  * Because this value is hyper-specific, there might only be 1 person in a dataset of 5,000 who matches it.
  * If that single person happened to reject the loan, the model calculates a strict 0% probability for that entire income level. This causes the model to become blindly rigid and perform terribly on unseen future data.
* **The Power of Post-Binning Probabilities**
  * Once the data is grouped, the query shifts to a much more reliable question: *"What is the probability that a customer in the 'High Income' bracket will accept a personal loan?"*.
  * Now, Naive Bayes has a robust pool of roughly 1,000 historical customers within that bracket to count and evaluate.
  * The resulting probability calculation is smooth, statistically significant, and highly generalizable to new customers.

---

####  3. SELECTING THE RIGHT STRATEGY BASED ON DATA DISTRIBUTION

Before slicing data into buckets, executing a quick `.describe()` method is equivalent to a doctor checking a patient's pulse. Based on that statistical distribution, we choose between two classic mathematical weapons:



* **Scenario A: Normal Distribution — Choosing `pd.cut` (Equal-Width Slicing)**
  * **Identification:** The data distribution is relatively symmetric, forming a classic bell-shaped curve where the distances between quartiles are uniform. Demographic Age is a prime example.
  * **Mechanism:** `pd.cut` acts like a standard physical ruler. It calculates the absolute distance between the absolute minimum and maximum values and chops that length into intervals of perfectly equal width (e.g., slicing exactly every 10 years).
  * **Visual Result:** The resulting ranges look highly intuitive and readable to human analysts (e.g., 20–30, 30–40, 40–50). While the sample count will naturally peak in the middle buckets and thin out at the extreme ends, it mirrors real societal structures accurately.
* **Scenario B: Skewed Distribution — Choosing `pd.qcut` (Equal-Frequency Slicing)**
  * **Identification:** Financial variables like annual Income or Credit Card Average (`CCAvg`) are inherently heavily skewed. The vast majority of the population clusters around low-to-mid tiers (bloating the bottom of the graph), while a tiny fraction of wealthy individuals stretches a long tail far out to the right.
  * **The Danger of Choosing Wrong:** If you rigidly apply `pd.cut` to split a wealth spectrum ranging from \$10,000 to \$1,000,000 into 5 equal parts, you will create **"empty buckets"** at the top tier. The \$800k–\$1M bucket might only contain 1 or 2 outlier millionaires, while the \$10k–\$200k bucket will be suffocatingly overcrowded. A Naive Bayes model can learn absolutely nothing from a category populated by only one person.
  * **Mechanism:** `pd.qcut` solves this by ignoring numerical distance entirely, focusing strictly on sample frequency. If you have 5,000 rows and want 5 bins, it counts exactly 1,000 rows, drops a boundary line, counts the next 1,000 rows, and drops another.
  * **Visual Result:** The numerical widths of the resulting intervals will look uneven (the low-income range will be very narrow, while the high-income range will stretch very wide). However, **the number of data points inside every single bucket is perfectly identical**. This ensures that the machine learning model has an equal, statistically viable foundation to calculate conditional probabilities fairly across all financial strata.

---

####  CONCLUSION
Data discretization is not just a routine syntax trick in a preprocessing pipeline; it is a vital strategic decision. By deliberately trading away raw numerical precision for robust behavioral segments, we liberate mathematical models like Naive Bayes from data noise—enabling smoother training cycles, preventing overfitting, and delivering highly actionable insights that drive real business revenue.

# 13.2. eBay Auctions — Boosting and Bagging

# 13.3. Predicting Delayed Flights (Boosting)

# 13.4. Hair Care Product — Uplift Modeling

#